<a href="https://colab.research.google.com/github/hbisgin/DeepLearning/blob/main/DL_15_RNN_Name_LangExample_w_Embedding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!wget https://raw.githubusercontent.com/EdwardRaff/Inside-Deep-Learning/refs/heads/main/idlmam.py

--2026-03-23 13:50:52--  https://raw.githubusercontent.com/EdwardRaff/Inside-Deep-Learning/refs/heads/main/idlmam.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 22064 (22K) [text/plain]
Saving to: ‘idlmam.py.2’

idlmam.py.2         100%[===================>]  21.55K  --.-KB/s    in 0.001s  

2026-03-23 13:50:53 (14.8 MB/s) - ‘idlmam.py.2’ saved [22064/22064]



In [ ]:
import sys
sys.path.append('/content/drive/MyDrive/DATA/')
sys.path.append('/content/idlmam.py')

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import transforms

from torch.utils.data import Dataset, DataLoader

from tqdm.autonotebook import tqdm

import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.pyplot import imshow

import pandas as pd

from sklearn.metrics import accuracy_score

from idlmam import train_simple_network, Flatten, weight_reset

In [ ]:
zip_file_url = "https://download.pytorch.org/tutorial/data.zip"

import requests, zipfile, io
r = requests.get(zip_file_url)
z = zipfile.ZipFile(io.BytesIO(r.content))
z.extractall()


In [ ]:
with open("/content/data/names/English.txt", "r") as f:
    for line in f:
        print(line.strip())

Abbas
Abbey
Abbott
Abdi
Abel
Abraham
Abrahams
Abrams
Ackary
Ackroyd
Acton
Adair
Adam
Adams
Adamson
Adanet
Addams
Adderley
Addinall
Addis
Addison
Addley
Aderson
Adey
Adkins
Adlam
Adler
Adrol
Adsett
Agar
Ahern
Aherne
Ahmad
Ahmed
Aikman
Ainley
Ainsworth
Aird
Airey
Aitchison
Aitken
Akhtar
Akram
Alam
Alanson
Alber
Albert
Albrighton
Albutt
Alcock
Alden
Alder
Aldersley
Alderson
Aldred
Aldren
Aldridge
Aldworth
Alesbury
Alexandar
Alexander
Alexnader
Alford
Algar
Ali
Alker
Alladee
Allam
Allan
Allard
Allaway
Allcock
Allcott
Alldridge
Alldritt
Allen
Allgood
Allington
Alliott
Allison
Allkins
Allman
Allport
Allsop
Allum
Allwood
Almond
Alpin
Alsop
Altham
Althoff
Alves
Alvey
Alway
Ambrose
Amesbury
Amin
Amner
Amod
Amor
Amos
Anakin
Anderson
Andersson
Anderton
Andrew
Andrews
Angus
Anker
Anley
Annan
Anscombe
Ansell
Anstee
Anthony
Antic
Anton
Antony
Antram
Anwar
Appleby
Appleton
Appleyard
Apsley
Arah
Archer
Ardern
Arkins
Armer
Armitage
Armour
Armsden
Armstrong
Arnall
Arnett
Arnold
Arnott
Arrowsmith
Arscott

#We will use some code to remove UNICODE tokens to make life easy for us processing wise
#e.g., convert something like "Ślusàrski" to Slusarski

In [ ]:
namge_language_data = {}
import unicodedata
import string

all_letters = string.ascii_letters + " .,;'"
n_letters = len(all_letters)
alphabet = {}
for i in range(n_letters):
    alphabet[all_letters[i]] = i

# Turn a Unicode string to plain ASCII (ref: https://stackoverflow.com/a/518232/2809427)
def unicodeToAscii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
        and c in all_letters
    )

#Loop through every language, open the zip file entry, and read all the lines from the text file.
for zip_path in z.namelist():
    if "data/names/" in zip_path and zip_path.endswith(".txt"):
        lang = zip_path[len("data/names/"):-len(".txt")]
        with z.open(zip_path) as myfile:
            lang_names = [unicodeToAscii(line).lower() for line in str(myfile.read(), encoding='utf-8').strip().split("\n")]
            namge_language_data[lang] = lang_names
        print(lang, ": ", len(lang_names)) #Print out the name of each language too.

Arabic :  2000
Chinese :  268
Czech :  519
Dutch :  297
English :  3668
French :  277
German :  724
Greek :  203
Irish :  232
Italian :  709
Japanese :  991
Korean :  94
Polish :  139
Portuguese :  74
Russian :  9408
Scottish :  100
Spanish :  298
Vietnamese :  73


#creating Dataset subclass specific to thir problem. Please note that every sequence will have a label, i.e., language

In [ ]:
class LanguageNameDataset(Dataset):

    def __init__(self, lang_name_dict, vocabulary):
        self.label_names = [x for x in lang_name_dict.keys()]
        self.data = []
        self.labels = []
        self.vocabulary = vocabulary
        for y, language in enumerate(self.label_names):
            for sample in lang_name_dict[language]:
                self.data.append(sample)
                self.labels.append(y)

    def __len__(self):
        return len(self.data)

    def string2InputVec(self, input_string):
        """
        This method will convert any input string into a vector of long values, according to the vocabulary used by this object.
        input_string: the string to convert to a tensor
        """
        T = len(input_string) #How many characters long is the string?

        #Create a new tensor to store the result in
        name_vec = torch.zeros((T), dtype=torch.long)
        #iterate through the string and place the appropriate values into the tensor
        for pos, character in enumerate(input_string):
            name_vec[pos] = self.vocabulary[character]

        return name_vec

    def __getitem__(self, idx):
        name = self.data[idx]
        label = self.labels[idx]

        #Convert the correct class label into a tensor for PyTorch
        #label_vec = torch.tensor([label], dtype=torch.long)

        return self.string2InputVec(name), label

In [ ]:
dataset = LanguageNameDataset(namge_language_data, alphabet)

train_data, test_data = torch.utils.data.random_split(dataset, (len(dataset)-300, 300))
train_loader = DataLoader(train_data, batch_size=1, shuffle=True)
test_loader = DataLoader(test_data, batch_size=1, shuffle=False)

In [ ]:
with torch.no_grad():
    input_sequence = torch.tensor([0, 1, 1, 0, 2], dtype=torch.long)
    embd = nn.Embedding(3, 2)
    x_seq = embd(input_sequence)
    print(input_sequence.shape, x_seq.shape)
    print(x_seq)

torch.Size([5]) torch.Size([5, 2])
tensor([[ 0.1344, -1.4789],
        [-0.7364, -0.7122],
        [-0.7364, -0.7122],
        [ 0.1344, -1.4789],
        [-0.1933,  1.3256]])


In [ ]:
class LastTimeStep(nn.Module):
    """
    A class for extracting the hidden activations of the last time step following
    the output of a PyTorch RNN module.
    """
    def __init__(self, rnn_layers=1, bidirectional=False):
        super(LastTimeStep, self).__init__()
        self.rnn_layers = rnn_layers
        if bidirectional:
            self.num_directions = 2
        else:
            self.num_directions = 1

    def forward(self, input):
        #Result is either a tupe (out, h_t)
        #or a tuple (out, (h_t, c_t))
        rnn_output = input[0]
        last_step = input[1] #this will be h_t
        if(type(last_step) == tuple):#unless it's a tuple,
            last_step = last_step[0]#then h_t is the first item in the tuple
        batch_size = last_step.shape[1] #per docs, shape is: '(num_layers * num_directions, batch, hidden_size)'
        #reshaping so that everything is separate
        last_step = last_step.view(self.rnn_layers, self.num_directions, batch_size, -1) #(S, D, B, H)
        #print(last_step.shape)
        #We want the last layer's results
        last_step = last_step[self.rnn_layers-1]
        #print(last_step.shape)
        #Re order so batch comes first
        last_step = last_step.permute(1, 0, 2) # (D, B, H) --> (B, D, H)
        #print(last_step.shape)
        #print(last_step.reshape(batch_size, -1).shape)
        #Finally, flatten the last two dimensions into one
        return last_step.reshape(batch_size, -1) # (B, D * H)

In [ ]:
#this part has the same code as the following code block but dimensions were confusin. So, I've corrected and added it in another code block
'''D = 64
vocab_size = len(all_letters)
hidden_nodes = 256
classes = len(dataset.label_names)

first_rnn = nn.Sequential(
  nn.Embedding(vocab_size, D), #(B, T) -> (B, T, D)
  nn.RNN(D, hidden_nodes, batch_first=True), #(B, T, D) -> ( (B,T,D) , (S, B, D)  )
  #the tanh activation is built into the RNN object, so we don't need to do it here
  LastTimeStep(), #We need to take the RNN output and reduce it to one item, (B, D)
  nn.Linear(hidden_nodes, classes), #(B, D) -> (B, classes)
)
'''

"D = 64\nvocab_size = len(all_letters)\nhidden_nodes = 256\nclasses = len(dataset.label_names)\n\nfirst_rnn = nn.Sequential(\n  nn.Embedding(vocab_size, D), #(B, T) -> (B, T, D)\n  nn.RNN(D, hidden_nodes, batch_first=True), #(B, T, D) -> ( (B,T,D) , (S, B, D)  )\n  #the tanh activation is built into the RNN object, so we don't need to do it here\n  LastTimeStep(), #We need to take the RNN output and reduce it to one item, (B, D)\n  nn.Linear(hidden_nodes, classes), #(B, D) -> (B, classes)\n)\n"

In [ ]:
D = 64
vocab_size = len(all_letters)
hidden_nodes = 256
classes = len(dataset.label_names)

first_rnn = nn.Sequential(
  nn.Embedding(vocab_size, D),                  # (B, T) -> (B, T, D)
  nn.RNN(D, hidden_nodes, batch_first=True),    # (B, T, D) -> ((B, T, H), (S, B, H))
  # tanh is built into nn.RNN
  LastTimeStep(),                               # ((B, T, H), (S, B, H)) -> (B, H)
  nn.Linear(hidden_nodes, classes),             # (B, H) -> (B, classes)
)

In [ ]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

In [ ]:
loss_func = nn.CrossEntropyLoss()
batch_one_train = train_simple_network(first_rnn, loss_func, train_loader, test_loader=test_loader, score_funcs={'Accuracy': accuracy_score}, device=device, epochs=1)

Epoch:   0%|          | 0/1 [00:00<?, ?it/s]

Training:   0%|          | 0/19774 [00:00<?, ?it/s]

KeyboardInterrupt: 

#please change the batch size and try to run your code. You are expected to get an error. Please submit your notebook with that error printed.